In [ ]:
# Databricks notebook source

# COMMAND ----------

%pip install boto3
dbutils.library.restartPython()

# COMMAND ----------

# =========================
# CONFIGURATION
# =========================
catalog     = "cabpluse360_team2"
gold_schema = "cabpulse360_gold_team2"

aws_key    = "***************"
aws_secret = "*************************************"
bucket     = "aws-macro-project"
region     = "us-east-1"  


# Volume path — we use the bronze schema's volume as temp storage
volume_path = f"/Volumes/{catalog}/cabpulse360_bronze_team2/gold_export"

# COMMAND ----------

# =========================
# CREATE VOLUME (run once)
# =========================
spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {catalog}.cabpulse360_bronze_team2.gold_export
""")
print("Volume ready.")

# COMMAND ----------

import boto3
import os

s3 = boto3.client(
    "s3",
    aws_access_key_id=aws_key,
    aws_secret_access_key=aws_secret,
    region_name=region
)

gold_tables = [
    "dim_date",
    "dim_vendor",
    "dim_taxi_zone",
    "dim_rate_code",
    "dim_payment_type",
    "dim_time_block",
    "fact_trips",
    "fact_daily_zone_summary"
]

# COMMAND ----------

# =========================
# EXPORT LOOP
# =========================
for table in gold_tables:
    print(f"\nExporting {table}...")

    # Step 1: Write single parquet file to Unity Catalog Volume
    write_path = f"{volume_path}/{table}"

    spark.table(f"{catalog}.{gold_schema}.{table}") \
        .coalesce(1) \
        .write \
        .mode("overwrite") \
        .parquet(write_path)

    # Step 2: Find the part file
    files = os.listdir(write_path)
    part_file = [f for f in files if f.startswith("part-") and f.endswith(".parquet")]

    if not part_file:
        print(f"  ERROR: No parquet file found for {table}")
        continue

    local_path = os.path.join(write_path, part_file[0])

    # Step 3: Upload to S3
    s3_key = f"Team2/gold_parquet/{table}.parquet"
    s3.upload_file(local_path, bucket, s3_key)

    size_kb = round(os.path.getsize(local_path) / 1024, 1)
    print(f"  Uploaded to s3://{bucket}/{s3_key} ({size_kb} KB)")

    # Step 4: Clean up volume temp file
    import shutil
    shutil.rmtree(write_path)
    print(f"  Temp files cleaned up")

print("\n\nAll Gold tables exported to S3.")

# COMMAND ----------

# =========================
# VERIFY
# =========================
print(f"Files in s3://{bucket}/cabpulse-macro/team2/gold_parquet/\n")

response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix="Team2/gold_parquet/"
)

for obj in response.get("Contents", []):
    size_kb = round(obj["Size"] / 1024, 1)
    print(f"  {obj['Key']}  ({size_kb} KB)")